# Finetuning using Quantized LoRA DPO

Uncomment below lines only on first run of a session to install libraries to speed up restarts on the same session. Based on the model, context length and other parameters that you set for training, there will be a lot of trial and error and so commenting this will speed up the process a lot.

Also, make sure that whenever you want to restart on Google Colab, you should restart the session. Do not delete runtime and then restart because that will take a lot of time to setup the environment again as well as needing the following line uncommented so that the right libraries can be installed again.


In [1]:
# %pip install torch
# %pip install -U transformers trl datasets accelerate evaluate sentencepiece bitsandbytes protobuf==3.20.3 wandb

The following 2 cells will set up the finetuning environment to properly log you in to your wandb and huggingface accounts for training logs as well as syncing with Huggingface.

In [2]:
from google.colab import userdata
import wandb
import os

# Login to your WandB account
wandb.login(userdata.get('WANDB_API_KEY'))

# Configure project settings
os.environ["WANDB_PROJECT"] = "gemma3-270m-orca-dpo"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: xebia-shriansh (xebia-shriansh-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

The following cell chooses the model that you are trying to finetune. For this example, gemma-3-270m-it model has been used since it is a very small model which can be finetuned very quickly with the Google Colab T4 GPU instance.

Note: Make sure that you are using bitsandbytes for quantization based on your requirement. Also, based on your model, you might have to comment out or do some customization for the following line: model.config.pad_token_id = tokenizer.pad_token_id



In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-3-270m-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="eager",
    quantization_config=bnb_config,
)
# model = model.unload()

tokenizer = AutoTokenizer.from_pretrained(model_id)
# tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Device: cuda:0
DType: torch.bfloat16


The following cell is used to choose the dataset on which you want to finetune your model. Make sure you are choosing the correct split for training. If you are training for a supervised task and not just for style adaptation, then validation and train splits will need to be created separately.

Also, make sure that the messages template that you are using is compatible with the model that you are finetuning.

In [5]:
from datasets import load_dataset

dataset = load_dataset("Intel/orca_dpo_pairs", split="train[:1000]")

# Format function to ensure columns match DPOTrainer expectations
def format_dpo_dataset(examples):
    return {
        # The 'prompt' is the context (User side)
        "prompt": [
            {"role": "user", "content": examples["question"]}
        ],
        # The 'chosen' is the winning answer (Assistant side)
        "chosen": [
            {"role": "assistant", "content": examples["chosen"]}
        ],
        # The 'rejected' is the losing answer (Assistant side)
        "rejected": [
            {"role": "assistant", "content": examples["rejected"]}
        ],
    }

# Apply formatting and take a small subset for a quick demo
dataset = dataset.map(format_dpo_dataset, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.05)

# Filters out datapoints which will not be within the chosen max_length for DPO training.
# Important to achieve good quality data which fits within max_length for training and testing.
def filter_length(example):
    p_len = len(tokenizer.encode(str(example['prompt'])))
    c_len = len(tokenizer.encode(str(example['chosen'])))
    r_len = len(tokenizer.encode(str(example['rejected'])))
    return (p_len + max(c_len, r_len)) <= 512

# Apply to both
dataset["train"] = dataset["train"].filter(filter_length)
dataset["test"] = dataset["test"].filter(filter_length)

print(f"Cleaned Train: {len(dataset['train'])} rows")
print(f"Cleaned Test: {len(dataset['test'])} rows")

Filter:   0%|          | 0/950 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

Cleaned Train: 624 rows
Cleaned Test: 35 rows


In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['chosen', 'rejected', 'prompt'],
        num_rows: 624
    })
    test: Dataset({
        features: ['chosen', 'rejected', 'prompt'],
        num_rows: 35
    })
})

# Direct Preference Optimization (DPO) Configuration

This section configures the **Direct Preference Optimization (DPO)** parameters. Unlike Supervised Fine-Tuning (SFT), DPO aligns the model's behavior with human preferences by comparing "Chosen" vs "Rejected" responses. The settings below are specifically optimized for the **NVIDIA T4 GPU** (16GB VRAM) and the **Gemma 3 270M** architecture.

---

## 1. DPO Alignment & Memory (`DPOConfig`)

These arguments manage the high VRAM overhead of DPO, which traditionally requires holding two models in memory.

| Argument | Setting | Purpose & Hardware Impact |
| --- | --- | --- |
| **`beta`** | `0.1` | **Alignment Strength:** Controls how much the model stays "anchored" to the base model.<br> $0.1$ is standard for Orca; it ensures the model learns preferences without losing its original language ability. |
| **`max_length`** | `512` | Defines the total limit for **Prompt + Response**. Essential for Orca pairs to ensure reasoning chains aren't truncated,<br> which would break the DPO loss calculation. |
| **`per_device_train_batch_size`** | `2` | **VRAM Safety:** Lowered from SFT because DPO tracks "Policy" and "Reference" log-probs simultaneously.<br> `2` is the stable limit for a 16GB T4. |
| **`gradient_accumulation_steps`** | `8` | Simulates a larger batch size. Effective Batch Size = $2 \times 8 = 16$. This provides the gradient stability<br> needed for the model to converge on a preference. |
| **`learning_rate`** | `5e-6` | **Critical:** DPO requires a much lower learning rate than SFT ($10\times$ to $40\times$ lower). <br>High LRs in DPO cause "Model Collapse" (repetitive or nonsensical output). |
| **`precompute_ref_log_probs`** | `True` | **Hardware Optimization:** Calculates reference model scores once at the start and clears them from active VRAM.<br> This is what makes DPO possible on a single T4. |
| **`optim`** | `"paged_adamw_8bit"` | A memory-efficient optimizer that handles VRAM spikes during the preference comparison phase. |

---

## 2. LoRA Hyperparameters (`LoraConfig`)

We use LoRA to steer the model's preferences. Since Gemma 3 270M is small, the adapter needs enough capacity to capture "style" and "logic" simultaneously.

* **`r` (Rank) = `32`**: A higher rank helps smaller models understand the "why" behind the chosen response.
* **`lora_alpha` = `64`**: Usually set to $2 \times r$. This scales the influence of the DPO adapters over the base model weights.
* **`target_modules` = `"all-linear"`**: Targets Attention and MLP layers. Essential for DPO so the model can learn both the **format** (Attention) and the **facts** (MLP) of the Orca dataset.
* **`lora_dropout` = `0.05`**: Provides a safety net against "Reward Hacking," where the model finds a statistical shortcut to the "Chosen" answer instead of learning the logic.

---

## 3. Evaluation & Checkpointing Strategy

DPO is highly prone to overfitting. Monitoring metrics in real-time is the only way to find the "Sweet Spot."

* **`eval_steps` = `5`**: Allows real-time tracking of **Reward Accuracy**.
* **`save_strategy` = `"steps"`**: Switched from `epoch` to `steps` (every 10). This is critical because the best model is often found **before** the final step.
* **Key Metric: `eval/rewards/accuracies`**: The percentage of times the model correctly preferred the 'Chosen' sample. Target is **0.80–0.90**. If it hits **1.0**, the model may be starting to overfit.
* **Key Metric: `eval/rewards/margins`**: Measures the "gap" in confidence between the good and bad answer. A widening, positive margin indicates successful alignment.

In [7]:
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

# Define the output directory for DPO checkpoints
adapter_path = "/content/gemma3-270m-orca-DPO-checkpoints-v1"

# 1. DPO Specific Configuration
training_args = DPOConfig(
    output_dir=adapter_path,
    max_length=512,               # Total length of prompt + response
    per_device_train_batch_size=2, # Reduced from 8: DPO holds two models in memory (Policy & Ref)
    per_device_eval_batch_size=2,  # Important for low VRAM. By default, this is 8.
    gradient_accumulation_steps=8, # Increased to maintain effective batch size
    learning_rate=5e-6,           # DPO usually requires a much lower LR (10x-40x lower than SFT)
    lr_scheduler_type="cosine",
    num_train_epochs=2,           # DPO is very efficient; 1-3 epochs is usually enough
    optim="paged_adamw_8bit",
    warmup_steps=100,

    # DPO Specific Hyperparameter
    beta=0.1,                     # The "alignment temperature" (0.1 is standard)

    # Memory & Speed
    gradient_checkpointing=True,
    precompute_ref_log_probs=True,

    # Logging & Evaluation
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,

    # Saving & Hub
    save_strategy="steps",
    save_steps=10,
    # save_total_limit=12,        # (OPTIONAL) Used to limit number of total saved checkpoints
    push_to_hub=False,
    # load_best_model_at_end=True,  # Automatically reload the best one based on eval. Make take a lot of VRAM so disable this if you face CUDA OOM errors.
    # metric_for_best_model="eval/rewards/accuracies", # Best = highest preference accuracy
    report_to="wandb",
    run_name="gemma3-dpo-alignment"
)

# 2. LoRA Configuration
# LoRA config is similar to the one used in SFT notebook
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

# 3. DPO Trainer Initialization
# The trainer automatically handles the 'Reference Model' creation from your base model
trainer = DPOTrainer(
    model=model,                  # Your loaded IT or SFT-tuned model
    ref_model=None,               # None = Trainer creates a frozen copy of 'model' automatically
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"] if "test" in dataset else None,
    processing_class=tokenizer,
    peft_config=lora_config,
)

Tokenizing train dataset:   0%|          | 0/624 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Computing reference log probs for train dataset:   0%|          | 0/312 [00:00<?, ?it/s]

Computing reference log probs for eval dataset:   0%|          | 0/18 [00:00<?, ?it/s]

In [8]:
trainer.train()
trainer.save_model(adapter_path)
wandb.finish()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss
5,0.677865,0.678029
10,0.689416,0.696889
15,0.697290,0.689959
20,0.683184,0.677366
25,0.691386,0.666210
30,0.685150,0.613882
35,0.610216,0.600637
40,0.552178,0.542426
45,0.475130,0.466308
50,0.371592,0.381462


wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-10)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-20)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-30)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-40)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-50)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-60)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-70)... Done. 0.2s
wandb: Adding directory to artifact (/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-78)... Done. 0.2s


eval/entropy,████▇▇▆▆▅▄▃▂▁▁▂▂
eval/logits/chosen,▇█▇▇▇▇▆▅▅▄▃▂▁▁▂▃
eval/logits/rejected,██▇▇▇▇▆▆▅▄▃▂▁▁▂▂
eval/logps/chosen,████▇█▇▆▆▅▄▃▂▂▁▁
eval/logps/rejected,██████▇▇▆▆▅▄▃▂▁▁
eval/loss,█████▇▇▆▅▄▃▃▂▂▁▁
eval/mean_token_accuracy,█▇▇▇█▇▅▅▅▆▅▅▂▁▃▂
eval/num_tokens,▁▂▂▂▃▃▄▄▅▅▆▆▇▇██
eval/rewards/accuracies,▂▁▃▁▃▆▆▆▇▇▇▇▇▇██
eval/rewards/chosen,████▇█▇▆▆▅▄▃▂▂▁▁
+21,...


### Model Selection
Looking at the model evaluation metrics on WandB, we do achieve a perfect accuracy of 1 on eval dataset. However, this is a bit of a red flag since this may be showcasing overfitting. So, we will choose the checkpoint at **60th** step since it still has a good **eval accuracy** of **91.667** along with a decent **margin of eval reward margins** of about **1.8**. These 2 metrics are very important in DPO. The margin in particular signifies that the model is becoming more confident in the chosen response compared to the rejected responses.

Also, make sure to check **eval/logps/rejected** and **eval/logps/chosen**. Both may look as if they are going negative. However, the difference from the first step to chosen step needs to be checked. If the difference in logps rewards is bigger than in logps chosen, then that means that the model is performing well. What this means is that effectively the model thinks that the chosen response has a higher probability than rejected one which signals proper alignment with Orca style.

The following code snippet merges the base model with the LoRA adapter that we trained before and then saves the merged model along with the tokenizer.

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

adapter_path = "/content/gemma3-270m-orca-DPO-checkpoints-v1/checkpoint-60"            # Choose which adapters to merge, otherwise defaults to latest
merged_model_path = "/content/gemma3-270m-orca-dpo-v1-merged-chkpt-60/"              # Location of merged model directory

# Load base model and tokenizer
base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Load and merge the PEFT adapters onto the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()                                      # Merges base model with adapter parameters that we trained before and then removes LoRA specific modules to return a standard transformer

# Save the merged model and its tokenizer
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)

print(f"Model merged and saved to {merged_model_path}. Final model vocabulary size: {model.config.vocab_size}")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model merged and saved to /content/gemma3-270m-orca-dpo-v1-merged-chkpt-60/. Final model vocabulary size: 262144


### Test the fine-tuned model

Let's compare your fine-tuned model performance against the base model! This is a good validation for your finetuned model and showcases how its output has changed based on your dataset.

In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display, Markdown

merged_model_path = "/content/gemma3-270m-orca-dpo-v1-merged-chkpt-60/"

# Create Transformers inference pipeline
merged_model = AutoModelForCausalLM.from_pretrained(merged_model_path, device_map="auto")
merged_tokenizer = AutoTokenizer.from_pretrained(merged_model_path)
pipe = pipeline("text-generation", model=merged_model, tokenizer=merged_tokenizer)
pipe_base = pipeline("text-generation", model=model_id, device_map="auto")

# Test a prompt
instruction = """Compare a manual toothbrush and an electric toothbrush.
Use a bulleted list to evaluate them based on cost-effectiveness, cleaning efficiency, and ease of use.
End with a recommendation."""

context = ""
inference_messages = [
    {"role": "user", "content": f"{instruction}\n{context}"},
]

prompt = merged_tokenizer.apply_chat_template(inference_messages, tokenize=False, add_generation_prompt=True)
output = pipe(prompt, max_new_tokens=512)
output_base = pipe_base(prompt, max_new_tokens=512)
model_output = output[0]['generated_text'][len(prompt):].strip()
model_output_base = output_base[0]['generated_text'][len(prompt):].strip()

display(Markdown(f"\nFine-tuned model output: \n\n{model_output}"))

display(Markdown(f"\n\nBase model output: \n\n{model_output_base}"))

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Fine-tuned model output: 

Okay, here's a comparison of a manual toothbrush and an electric toothbrush, focusing on cost-effectiveness, cleaning efficiency, and ease of use:

*   **Cost-Effectiveness:**
    *   **Manual Toothbrush:** Generally more affordable, especially for basic brushing.
    *   **Electric Toothbrush:** More expensive, potentially costing more due to the battery, charging, and potentially more complex features.

*   **Cleaning Efficiency:**
    *   **Manual Toothbrush:** Can be less effective at removing plaque and bacteria, especially at the hard-to-reach areas.
    *   **Electric Toothbrush:** More effective at removing plaque and bacteria, leading to a more thorough cleaning.

*   **Ease of Use:**
    *   **Manual Toothbrush:** Generally easier to use for beginners and those with limited dexterity.
    *   **Electric Toothbrush:** Requires more dexterity and can be more challenging to master.

*   **Advantages:**
    *   **Manual Toothbrush:** Can be more convenient for those with mobility limitations or those who prefer a more hands-free experience.
    *   **Electric Toothbrush:** Offers a more comprehensive cleaning experience, potentially removing more plaque and bacteria.

*   **Disadvantages:**
    *   **Manual Toothbrush:** Can be more expensive than electric toothbrushes.
    *   **Electric Toothbrush:** Requires a charging station and can be more expensive.

**Recommendation:**

For most people, a **manual toothbrush is the better choice.** It offers a more affordable and convenient option for basic brushing, making it a more accessible and practical choice for those who are new to manual brushing. The ease of use and the ability to clean more effectively make it a worthwhile investment for those who prioritize convenience and have limited dexterity.



Base model output: 

Okay, here's a comparison of a manual toothbrush and an electric toothbrush, based on cost-effectiveness, cleaning efficiency, and ease of use:

*   **Cost-Effectiveness:**
    *   Manual toothbrushes are generally more affordable than electric toothbrushes.
    *   Electric toothbrushes can be more expensive, but often offer better value for money.
    *   The cost of a manual toothbrush can be lower, especially for basic cleaning tasks.

*   **Cleaning Efficiency:**
    *   Manual toothbrushes are generally more efficient in cleaning because they use fewer bristles and are more gentle.
    *   Electric toothbrushes can be more effective in cleaning, especially for certain types of plaque and stains.
    *   Electric toothbrushes can be more energy-efficient, saving you money on electricity.

*   **Ease of Use:**
    *   Manual toothbrushes are easier to use for beginners, as they require minimal manual effort.
    *   Electric toothbrushes can be more challenging to learn, requiring some initial practice.
    *   Electric toothbrushes are generally more user-friendly, with a simpler interface and fewer steps.

*   **Advantages:**
    *   Manual toothbrushes are generally more affordable.
    *   Electric toothbrushes can be more efficient for cleaning, especially for certain types of plaque and stains.
    *   Electric toothbrushes can be more energy-efficient.

*   **Disadvantages:**
    *   Manual toothbrushes can be less effective in cleaning due to the need for more manual effort.
    *   Electric toothbrushes can be more challenging to learn.
    *   Electric toothbrushes can be more expensive.

**Recommendation:**

For most people, a **manual toothbrush** is the better choice. While electric toothbrushes offer excellent cleaning capabilities, the cost-effectiveness and efficiency are often comparable or even better. The ease of use and efficiency make them a more practical option for most users.

As can be seen, the finetuned model using DPO is more concise and follows the Orca style of response.

Please note that the answer themselves might not be as good as larger models because these models still are very small at only 270 million parameters. However the finetuning process itself is a success due to the percievable change in response style.

Finally, this block pushes your model to huggingface and saves it there. This is especially important with temporary instances such as the Colab T4 instance so that you may use your model later.

In [29]:
from huggingface_hub import ModelCard, ModelCardData, whoami

model_name = "orca-DPO"

username = whoami()['name']
hf_repo_id = f"{username}/{model_name}-gemma-3-270m-it"

repo_url = merged_model.push_to_hub(hf_repo_id, commit_message="Upload model")
tokenizer.push_to_hub(hf_repo_id)

card_content = f"""
---
base_model: {model_id}
tags:
- text-generation
- Orca
- DPO FT
- gemma
---
A fine-tuned model based on `{model_id}` on the orca_dpo_pairs dataset by Intel for testing DPO FT."""
card = ModelCard(card_content)
card.push_to_hub(hf_repo_id)

print(f"Uploaded to {repo_url}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...qldffo6/model.safetensors:  30%|##9       |  160MB /  536MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp7pha0swe/tokenizer.json:  99%|#########9| 33.1MB / 33.4MB            

Uploaded to https://huggingface.co/Shriansh-Xebia/orca-DPO-gemma-3-270m-it/commit/3c6944377dbc03d456c6e48b2c2da474885d43b4
